In [9]:
library(tidyverse)
data("starwars", package = "dplyr")


glimpse(starwars)
# 87 Observations/Rows
# 14 Variables/Columns


Rows: 87
Columns: 14
$ name       <chr> "Luke Skywalker", "C-3PO", "R2-D2", "Darth Vader", "Leia Or…
$ height     <int> 172, 167, 96, 202, 150, 178, 165, 97, 183, 182, 188, 180, 2…
$ mass       <dbl> 77.0, 75.0, 32.0, 136.0, 49.0, 120.0, 75.0, 32.0, 84.0, 77.…
$ hair_color <chr> "blond", NA, NA, "none", "brown", "brown, grey", "brown", N…
$ skin_color <chr> "fair", "gold", "white, blue", "white", "light", "light", "…
$ eye_color  <chr> "blue", "yellow", "red", "yellow", "brown", "blue", "blue",…
$ birth_year <dbl> 19.0, 112.0, 33.0, 41.9, 19.0, 52.0, 47.0, NA, 24.0, 57.0, …
$ sex        <chr> "male", "none", "none", "male", "female", "male", "female",…
$ gender     <chr> "masculine", "masculine", "masculine", "masculine", "femini…
$ homeworld  <chr> "Tatooine", "Tatooine", "Naboo", "Tatooine", "Alderaan", "T…
$ species    <chr> "Human", "Droid", "Droid", "Human", "Human", "Human", "Huma…
$ films      <list> <"A New Hope", "The Empire Strikes Back", "Return of the J…
$ vehicles   <list>

In [2]:
starwars %>%
  summarise(
    height_NA = sum(is.na(height)),
    mass_NA = sum(is.na(mass)),
    homeworld_NA = sum(is.na(homeworld))
  )

height_NA,mass_NA,homeworld_NA
<int>,<int>,<int>
6,28,10


In [20]:
starwars_wide <- starwars %>%
  #Keep only rows with non-missing species
  filter(!is.na(species)) %>%
  
  #Group by species and gender
  group_by(species, gender) %>%
  
  #Compute mean height per group
  summarise(mean_height = mean(height, na.rm = TRUE), .groups = "drop") %>%
  
  #Pivot wider: species as rows, gender as columns
  pivot_wider(
    names_from = gender,
    values_from = mean_height
  )

starwars_wide

species,masculine,feminine
<chr>,<dbl>,<dbl>
Aleena,79.0000,NA
Besalisk,198.0000,NA
Cerean,198.0000,NA
Chagrian,196.0000,NA
Clawdite,NA,168.0000
Droid,140.0000,96.0000
Dug,112.0000,NA
Ewok,88.0000,NA
Geonosian,183.0000,NA


In [25]:
#Pivot Longer: species as rows, mean height/gender as columns
starwars_long <- starwars_wide %>%  
    pivot_longer(
        cols = -species,
        names_to = "gender",
        values_to = "mean_height"
      ) %>%
    filter(!is.na(mean_height))

starwars_long

species,gender,mean_height
<chr>,<chr>,<dbl>
Aleena,masculine,79.0000
Besalisk,masculine,198.0000
Cerean,masculine,198.0000
Chagrian,masculine,196.0000
Clawdite,feminine,168.0000
Droid,masculine,140.0000
Droid,feminine,96.0000
Dug,masculine,112.0000
Ewok,masculine,88.0000


In [27]:
starwars_bmi <- starwars %>%
  mutate(
    # BMI = mass / (height/100)^2
    BMI = mass / ( (height/100)^2 ),
    
    # Height category
    height_category = case_when(
      height < 170            ~ "short",
      height >= 170 & height <= 189 ~ "average",
      height >= 190           ~ "tall",
      TRUE                    ~ NA_character_  # keep NA if height is missing
    ),
    
    # Replace missing homeworld with "Unknown"
    homeworld = if_else(is.na(homeworld), "Unknown", homeworld)
  )

glimpse(starwars_bmi)

Rows: 87
Columns: 16
$ name            <chr> "Luke Skywalker", "C-3PO", "R2-D2", "Darth Vader", "Le…
$ height          <int> 172, 167, 96, 202, 150, 178, 165, 97, 183, 182, 188, 1…
$ mass            <dbl> 77.0, 75.0, 32.0, 136.0, 49.0, 120.0, 75.0, 32.0, 84.0…
$ hair_color      <chr> "blond", NA, NA, "none", "brown", "brown, grey", "brow…
$ skin_color      <chr> "fair", "gold", "white, blue", "white", "light", "ligh…
$ eye_color       <chr> "blue", "yellow", "red", "yellow", "brown", "blue", "b…
$ birth_year      <dbl> 19.0, 112.0, 33.0, 41.9, 19.0, 52.0, 47.0, NA, 24.0, 5…
$ sex             <chr> "male", "none", "none", "male", "female", "male", "fem…
$ gender          <chr> "masculine", "masculine", "masculine", "masculine", "f…
$ homeworld       <chr> "Tatooine", "Tatooine", "Naboo", "Tatooine", "Alderaan…
$ species         <chr> "Human", "Droid", "Droid", "Human", "Human", "Human", …
$ films           <list> <"A New Hope", "The Empire Strikes Back", "Return of …
$ vehicles        <

In [28]:

starwars_bmi_display <- starwars_bmi %>%
  mutate(
    films = sapply(films, paste, collapse = ", "),
    species = sapply(species, paste, collapse = ", "),
    vehicles = sapply(vehicles, paste, collapse = ", "),
    starships = sapply(starships, paste, collapse = ", ")
  )

# Display with rich
starwars_bmi_display

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships,BMI,height_category
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"A New Hope, The Empire Strikes Back, Return of the Jedi, Revenge of the Sith, The Force Awakens","Snowspeeder, Imperial Speeder Bike","X-wing, Imperial shuttle",26.02758,average
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"A New Hope, The Empire Strikes Back, Return of the Jedi, The Phantom Menace, Attack of the Clones, Revenge of the Sith",,,26.89232,short
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"A New Hope, The Empire Strikes Back, Return of the Jedi, The Phantom Menace, Attack of the Clones, Revenge of the Sith, The Force Awakens",,,34.72222,short
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"A New Hope, The Empire Strikes Back, Return of the Jedi, Revenge of the Sith",,TIE Advanced x1,33.33007,tall
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"A New Hope, The Empire Strikes Back, Return of the Jedi, Revenge of the Sith, The Force Awakens",Imperial Speeder Bike,,21.77778,short
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"A New Hope, Attack of the Clones, Revenge of the Sith",,,37.87401,average
Beru Whitesun Lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"A New Hope, Attack of the Clones, Revenge of the Sith",,,27.54821,short
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,,34.00999,short
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing,25.08286,average


In [29]:
starwars %>%
  select(name, films) %>%              # keep only relevant columns
  unnest_longer(films) %>%             # convert list-column into long format
  group_by(name) %>%                    # group by character
  summarise(film_count = n(), .groups = "drop") %>%  # count number of films
  arrange(desc(film_count)) %>%         # sort descending
  slice_head(n = 8)     

name,film_count
<chr>,<int>
R2-D2,7
C-3PO,6
Obi-Wan Kenobi,6
Chewbacca,5
Leia Organa,5
Luke Skywalker,5
Palpatine,5
Yoda,5


Because long data makes it easier to work with variables and observations by having 1:1 relation/ratio of data, we can use tiderveryse tools to get our desired output from those data that will be harder if they are not in the long format. Most plotting and analysis is also easier and more digestible to be understood by functions and readable by humans due to less indexing/wordy or long values.